In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Customer Churn Prediction - Interactive Notebook\n",
    "\n",
    "This notebook provides an interactive environment to explore the customer churn prediction model.\n",
    "\n",
    "## Project Goals\n",
    "- Predict customer churn with 92%+ accuracy\n",
    "- Implement feature engineering techniques\n",
    "- Optimize model through hyperparameter tuning\n",
    "- Evaluate model performance comprehensively"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "sys.path.append('../src')\n",
    "\n",
    "from churn_prediction import ChurnPredictor\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Initialize and Load Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize the predictor\n",
    "predictor = ChurnPredictor()\n",
    "\n",
    "# Create sample data\n",
    "df = predictor.create_sample_data(n_samples=5000)\n",
    "\n",
    "print(f\"Dataset shape: {df.shape}\")\n",
    "print(f\"\\nChurn rate: {df['churn'].mean():.2%}\")\n",
    "print(f\"\\nFirst few rows:\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Exploratory Data Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Basic statistics\n",
    "df.describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for missing values\n",
    "print(\"Missing values:\")\n",
    "print(df.isnull().sum())\n",
    "\n",
    "# Data types\n",
    "print(\"\\nData types:\")\n",
    "print(df.dtypes)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Churn distribution\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Pie chart\n",
    "churn_counts = df['churn'].value_counts()\n",
    "axes[0].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', startangle=90, colors=['#2ecc71', '#e74c3c'])\n",
    "axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')\n",
    "\n",
    "# Bar chart by contract type\n",
    "churn_by_contract = df.groupby('contract_type')['churn'].mean() * 100\n",
    "churn_by_contract.plot(kind='bar', ax=axes[1], color='coral')\n",
    "axes[1].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')\n",
    "axes[1].set_ylabel('Churn Rate (%)')\n",
    "axes[1].set_xlabel('Contract Type')\n",
    "axes[1].tick_params(axis='x', rotation=45)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation analysis\n",
    "numeric_cols = df.select_dtypes(include=[np.number]).columns\n",
    "correlation = df[numeric_cols].corr()\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)\n",
    "plt.title('Correlation Matrix', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of key features\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "# Age distribution\n",
    "axes[0, 0].hist(df['age'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')\n",
    "axes[0, 0].set_title('Age Distribution', fontweight='bold')\n",
    "axes[0, 0].set_xlabel('Age')\n",
    "axes[0, 0].set_ylabel('Frequency')\n",
    "\n",
    "# Tenure distribution\n",
    "axes[0, 1].hist(df['tenure_months'], bins=30, edgecolor='black', alpha=0.7, color='lightgreen')\n",
    "axes[0, 1].set_title('Tenure Distribution', fontweight='bold')\n",
    "axes[0, 1].set_xlabel('Tenure (months)')\n",
    "axes[0, 1].set_ylabel('Frequency')\n",
    "\n",
    "# Monthly charges distribution\n",
    "axes[1, 0].hist(df['monthly_charges'], bins=30, edgecolor='black', alpha=0.7, color='salmon')\n",
    "axes[1, 0].set_title('Monthly Charges Distribution', fontweight='bold')\n",
    "axes[1, 0].set_xlabel('Monthly Charges ($)')\n",
    "axes[1, 0].set_ylabel('Frequency')\n",
    "\n",
    "# Customer service calls\n",
    "df['customer_service_calls'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='plum')\n",
    "axes[1, 1].set_title('Customer Service Calls Distribution', fontweight='bold')\n",
    "axes[1, 1].set_xlabel('Number of Calls')\n",
    "axes[1, 1].set_ylabel('Frequency')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Feature Engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Apply feature engineering\n",
    "df_engineered = predictor.feature_engineering(df)\n",
    "\n",
    "print(f\"Original features: {df.shape[1]}\")\n",
    "print(f\"After feature engineering: {df_engineered.shape[1]}\")\n",
    "print(f\"\\nNew features created:\")\n",
    "new_features = set(df_engineered.columns) - set(df.columns)\n",
    "for feature in new_features:\n",
    "    print(f\"  - {feature}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# View the new features\n",
    "df_engineered[['customer_id', 'avg_monthly_charge', 'tenure_group', 'charge_per_service', 'high_risk', 'service_engagement']].head(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze engineered features\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# High-risk customers\n",
    "high_risk_churn = df_engineered.groupby('high_risk')['churn'].mean() * 100\n",
    "high_risk_churn.plot(kind='bar', ax=axes[0], color=['green', 'red'])\n",
    "axes[0].set_title('Churn Rate: High-Risk vs Low-Risk Customers', fontweight='bold')\n",
    "axes[0].set_xlabel('High Risk (0=No, 1=Yes)')\n",
    "axes[0].set_ylabel('Churn Rate (%)')\n",
    "axes[0].tick_params(axis='x', rotation=0)\n",
    "\n",
    "# Tenure group analysis\n",
    "tenure_churn = df_engineered.groupby('tenure_group')['churn'].mean() * 100\n",
    "tenure_churn.plot(kind='bar', ax=axes[1], color='steelblue')\n",
    "axes[1].set_title('Churn Rate by Tenure Group', fontweight='bold')\n",
    "axes[1].set_xlabel('Tenure Group')\n",
    "axes[1].set_ylabel('Churn Rate (%)')\n",
    "axes[1].tick_params(axis='x', rotation=45)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Data Preparation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "# Split the data\n",
    "train_df, test_df = train_test_split(\n",
    "    df_engineered, \n",
    "    test_size=0.2, \n",
    "    random_state=42, \n",
    "    stratify=df_engineered['churn']\n",
    ")\n",
    "\n",
    "print(f\"Training set: {train_df.shape}\")\n",
    "print(f\"Test set: {test_df.shape}\")\n",
    "print(f\"\\nTrain churn rate: {train_df['churn'].mean():.2%}\")\n",
    "print(f\"Test churn rate: {test_df['churn'].mean():.2%}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Preprocess\n",
    "X_train, y_train = predictor.preprocess_data(train_df, is_training=True)\n",
    "X_test, y_test = predictor.preprocess_data(test_df, is_training=False)\n",
    "\n",
    "print(f\"X_train shape: {X_train.shape}\")\n",
    "print(f\"X_test shape: {X_test.shape}\")\n",
    "print(f\"\\nFeatures: {list(X_train.columns)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Model Training\n",
    "\n",
    "**Note:** Hyperparameter tuning can take 5-10 minutes. For faster execution, set `tune_hyperparameters=False`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train the model\n",
    "# Set tune_hyperparameters=True for best results (takes longer)\n",
    "# Set tune_hyperparameters=False for quick testing\n",
    "\n",
    "print(\"Training model...\")\n",
    "print(\"⚠️ Note: With hyperparameter tuning, this may take 5-10 minutes\")\n",
    "\n",
    "predictor.train_model(X_train, y_train, tune_hyperparameters=False)  # Change to True for full tuning\n",
    "\n",
    "print(\"\\n✅ Training completed!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Model Evaluation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate the model\n",
    "results = predictor.evaluate_model(X_test, y_test)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get predictions\n",
    "y_pred = predictor.model.predict(X_test)\n",
    "y_pred_proba = predictor.model.predict_proba(X_test)[:, 1]\n",
    "\n",
    "# Create a comparison dataframe\n",
    "comparison = pd.DataFrame({\n",
    "    'Actual': y_test.values,\n",
    "    'Predicted': y_pred,\n",
    "    'Probability': y_pred_proba\n",
    "})\n",
    "\n",
    "print(\"Sample predictions:\")\n",
    "comparison.head(20)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Feature Importance Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display top features\n",
    "print(\"Top 15 Most Important Features:\")\n",
    "print(predictor.feature_importance.head(15))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize feature importance\n",
    "plt.figure(figsize=(10, 8))\n",
    "top_15 = predictor.feature_importance.head(15)\n",
    "plt.barh(range(len(top_15)), top_15['importance'], color='teal')\n",
    "plt.yticks(range(len(top_15)), top_15['feature'])\n",
    "plt.xlabel('Importance Score', fontsize=12)\n",
    "plt.title('Top 15 Feature Importances', fontsize=14, fontweight='bold')\n",
    "plt.gca().invert_yaxis()\n",
    "plt.grid(axis='x', alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Comprehensive Visualizations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate all visualizations\n",
    "predictor.plot_results(X_test, y_test, save_path='../models/')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Making Predictions on New Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Example: Predict churn for new customers\n",
    "new_customers = pd.DataFrame({\n",
    "    'customer_id': [99999, 88888, 77777],\n",
    "    'age': [35, 55, 28],\n",
    "    'tenure_months': [6, 48, 2],\n",
    "    'monthly_charges': [120.0, 75.0, 95.0],\n",
    "    'total_charges': [720.0, 3600.0, 190.0],\n",
    "    'contract_type': ['Month-to-month', 'Two year', 'Month-to-month'],\n",
    "    'payment_method': ['Electronic check', 'Credit card', 'Electronic check'],\n",
    "    'internet_service': ['Fiber optic', 'DSL', 'Fiber optic'],\n",
    "    'online_security': ['No', 'Yes', 'No'],\n",
    "    'tech_support': ['No', 'Yes', 'No'],\n",
    "    'num_services': [2, 5, 1],\n",
    "    'customer_service_calls': [7, 2, 8]\n",
    "})\n",
    "\n",
    "print(\"New customers to predict:\")\n",
    "print(new_customers)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Apply feature engineering\n",
    "new_customers_engineered = predictor.feature_engineering(new_customers)\n",
    "\n",
    "# Preprocess\n",
    "X_new, _ = predictor.preprocess_data(new_customers_engineered, is_training=False)\n",
    "\n",
    "# Predict\n",
    "predictions = predictor.model.predict(X_new)\n",
    "probabilities = predictor.model.predict_proba(X_new)[:, 1]\n",
    "\n",
    "# Display results\n",
    "results_df = pd.DataFrame({\n",
    "    'Customer_ID': new_customers['customer_id'],\n",
    "    'Prediction': ['CHURN' if p == 1 else 'NO CHURN' for p in predictions],\n",
    "    'Churn_Probability': [f\"{p:.2%}\" for p in probabilities],\n",
    "    'Risk_Level': ['HIGH' if p > 0.7 else 'MEDIUM' if p > 0.4 else 'LOW' for p in probabilities]\n",
    "})\n",
    "\n",
    "print(\"\\nPrediction Results:\")\n",
    "print(\"=\"*60)\n",
    "print(results_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Save the Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save the trained model\n",
    "predictor.save_model('../models/churn_model.pkl')\n",
    "print(\"✅ Model saved successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Load and Reuse Saved Model (Optional)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Example: Load a previously saved model\n",
    "new_predictor = ChurnPredictor()\n",
    "new_predictor.load_model('../models/churn_model.pkl')\n",
    "\n",
    "# Now you can use it to make predictions\n",
    "print(\"✅ Model loaded successfully!\")\n",
    "print(f\"\\nModel type: {type(new_predictor.model)}\")\n",
    "print(f\"Number of features: {len(new_predictor.feature_importance)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Summary and Insights\n",
    "\n",
    "### Key Findings:\n",
    "\n",
    "1. **Model Performance**\n",
    "   - The Random Forest model achieves high accuracy in predicting customer churn\n",
    "   - Both precision and recall are balanced, making it suitable for production use\n",
    "\n",
    "2. **Important Features**\n",
    "   - Contract type is a strong predictor (month-to-month customers churn more)\n",
    "   - Tenure is critical (new customers are at higher risk)\n",
    "   - Customer service interactions indicate dissatisfaction\n",
    "   - Monthly charges impact churn decisions\n",
    "\n",
    "3. **Business Recommendations**\n",
    "   - Target month-to-month customers with long-term contract incentives\n",
    "   - Implement early intervention programs for new customers (first 12 months)\n",
    "   - Improve customer service quality to reduce complaints\n",
    "   - Consider pricing strategies for high-charge customers\n",
    "\n",
    "4. **Next Steps**\n",
    "   - Deploy the model in production\n",
    "   - Monitor model performance over time\n",
    "   - Retrain with new data periodically\n",
    "   - A/B test retention strategies based on predictions\n",
    "\n",
    "---\n",
    "\n",
    "**Congratulations! You've successfully built a customer churn prediction model!** 🎉"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}